### Chargement du DataFrame

In [ ]:
import pandas as pd
ENTRAINEMENT = ""

if ENTRAINEMENT == "regex":
    df = pd.read_csv("../data/processed/dataset_annotated_regex.csv")
else:
    df = pd.read_csv("../data/processed/dataset_avis.csv")


labels = ["qualité produit", "service livraison", "service client"]

df_annotated = df[df[labels].sum(axis=1) > 0]
df.shape

### Distribution des classes

In [ ]:
class_distribution = df_annotated[labels].sum().to_frame(name="nb_commentaires")
class_distribution["pourcentage"] = (
    class_distribution["nb_commentaires"] / len(df_annotated) * 100
)

class_distribution

### Affichage d'avis au hasard

In [ ]:
samples = []

pd.set_option("display.max_colwidth", None)

df_annotated = df_annotated.drop(columns=["client"], errors="ignore")

for label in labels:
    sample_df = df_annotated[df_annotated[label] == 1].sample(
        n=3,
        random_state=None
    )
    
    sample_df = sample_df.copy()
    sample_df["label_cible"] = label  # pour savoir pourquoi il est sélectionné
    
    samples.append(sample_df)

result = pd.concat(samples)

display_cols = result.rename(columns={
    "qualité produit": "produit",
    "service livraison": "livraison",
    "service client": "client"
})

display_cols = display_cols[[
    "label_cible",
    "produit",
    "livraison",
    "client",
    "clean_comment"
]]

display(display_cols)

# for _, row in display_cols.iterrows():
#     print("─" * 100)
#     print(f"Label cible : {row['label_cible']}")
#     print(f"Produit    : {row['produit']}")
#     print(f"Livraison  : {row['livraison']}")
#     print(f"Client    : {row['client']}")
#     print("\nCommentaire :")
#     print(row["clean_comment"])

### Création des variables X_test/y

In [ ]:
X_text = df_annotated["clean_comment"].values
y = df_annotated[[
    "qualité produit",
    "service livraison",
    "service client"
]].values

### Encoding de X_text

In [ ]:
from sentence_transformers import SentenceTransformer

model_emb = SentenceTransformer(
    "dangvantuan/french-document-embedding",
    trust_remote_code=True
)

X_embeddings = model_emb.encode(
    X_text,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

### Séparation des données en train/test

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

indices = np.arange(len(df_annotated))

X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X_embeddings,
    y,
    indices,
    test_size=0.2,
    random_state=42
)

### Chargement ou création du modèle

In [ ]:
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from joblib import dump, load

if ENTRAINEMENT == "regex":
    MODEL_PATH = Path("../models/LROneVsRestClassifier_regex.joblib")
else:
    MODEL_PATH = Path("../models/LROneVsRestClassifier.joblib")

if MODEL_PATH.exists():
    print("Modèle trouvé : chargement")
    clf = load(MODEL_PATH)

else:
    print("Aucun modèle trouvé : entraînement")

    log_reg = LogisticRegression(
        penalty="l2",
        solver="lbfgs",
        max_iter=1000,
        class_weight="balanced"
    )

    clf = OneVsRestClassifier(log_reg)
    clf.fit(X_train, y_train)

    MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    dump(clf, MODEL_PATH)

    print("Modèle entraîné et sauvegardé")

### CrossVal

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, f1_score

scorer = make_scorer(f1_score, average="micro")

cv_scores = cross_val_score(
    clf,
    X_embeddings,
    y,
    cv=5,
    scoring=scorer,
    n_jobs=-1
)

print("F1 micro par fold :", cv_scores)
print("F1 micro moyen   :", cv_scores.mean())
print("Écart-type       :", cv_scores.std())

### Calcul des probas sur test ou données labelisées

In [ ]:
EVAL_MODE = "test"

if EVAL_MODE == "test":
    print("Évaluation sur le jeu de test")

    X_eval = X_test
    y_eval = y_test
elif EVAL_MODE == "labellise":
    print("Évaluation sur les données labelisées")

    df_eval = pd.read_csv("../data/processed/100_avis_annote.csv", sep=";")

    X_eval_text = df_eval["clean_comment"].values
    y_eval = df_eval[[
        "qualité produit",
        "service livraison",
        "service client"
    ]].values
    X_eval = model_emb.encode(
        X_eval_text,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

y_proba = clf.predict_proba(X_eval)

### Affichage des résultats

In [ ]:
from sklearn.metrics import classification_report

threshold = 0.5

y_pred = (y_proba >= threshold).astype(int)

print(classification_report(
    y_eval,
    y_pred,
    target_names=[
        "qualité produit",
        "service livraison",
        "service client"
    ]
))

In [ ]:
import numpy as np

labels = ["qualité produit", "service livraison", "service client"]

threshold = 0.50

thresholds = {
    "qualité produit": threshold,
    "service livraison": threshold,
    "service client": threshold
}

y_pred = np.zeros_like(y_proba, dtype=int)

for i, label in enumerate(labels):
    y_pred[:, i] = (y_proba[:, i] >= thresholds[label]).astype(int)

from sklearn.metrics import confusion_matrix

for i, label in enumerate(labels):
    tn, fp, fn, tp = confusion_matrix(
        y_eval[:, i],
        y_pred[:, i]
    ).ravel()
    
    df_cm = pd.DataFrame(
        [[tn, fp], [fn, tp]],
        index=["Vrai 0", "Vrai 1"],
        columns=["Prédit 0", "Prédit 1"]
    )
    
    print(f"\n{label}")
    display(df_cm)



In [ ]:
if EVAL_MODE == "test":
    df_eval = df_annotated.iloc[idx_test].copy()
elif EVAL_MODE == "labeled":
    df_eval = df_eval.copy()
pd.set_option("display.max_colwidth", None)
for i, label in enumerate(labels):
    df_eval[f"y_true_{label}"] = y_eval[:, i]
    df_eval[f"y_pred_{label}"] = y_pred[:, i]
    df_eval[f"proba_{label}"] = y_proba[:, i]

# Affichage de quelques avis (faux négatifs ou faux positifs)
def show_errors(df, label, n=10):
    y_true = f"y_true_{label}"
    y_pred = f"y_pred_{label}"

    proba_cols = [f"proba_{l}" for l in labels]

    fn = df[(df[y_true] == 1) & (df[y_pred] == 0)]
    fp = df[(df[y_true] == 0) & (df[y_pred] == 1)]

    print(f"\n{label.upper()} — Faux négatifs")
    display(fn[["clean_comment"] + proba_cols].head(n))

    print(f"\n{label.upper()} — Faux positifs")
    display(fp[["clean_comment"] + proba_cols].head(n))

for label in labels:
    show_errors(df_eval, label, n=10)